In [10]:
import pandas as pd
import os
klasor_yolu = r"C:\Users\Tugce\OneDrive\Masaüstü\Proje_Omnichannel_DW" 

dosyalar= [
    "distribution_centers.csv",
    "products_old.csv",
    "users_old.csv",
    "orders.csv",
    "order_items.csv",
    "inventory_items_old.csv",
    "events_old.csv"]

for dosya in dosyalar:
    tam_yol = os.path.join(klasor_yolu, dosya)
    if os.path.exists(tam_yol):
        df_ornek =pd.read_csv(tam_yol, nrows=5)
        print(df_ornek,5)
    else:
        print(f" {dosya} bulunamadı\n")

   id            name  latitude  longitude
0   1      Memphis TN   35.1174   -89.9711
1   2      Chicago IL   41.8369   -87.6847
2   3      Houston TX   29.7604   -95.3698
3   4  Los Angeles CA   34.0500  -118.2500
4   5  New Orleans LA   29.9500   -90.0667 5
      id     cost     category  \
0  13842  2.51875  Accessories   
1  13928  2.33835  Accessories   
2  14115  4.87956  Accessories   
3  14157  4.64877  Accessories   
4  14273  6.50793  Accessories   

                                                name brand  retail_price  \
0   Low Profile Dyed Cotton Twill Cap - Navy W39S55D    MG          6.25   
1  Low Profile Dyed Cotton Twill Cap - Putty W39S55D    MG          5.95   
2       Enzyme Regular Solid Army Caps-Black W35S45D    MG         10.99   
3  Enzyme Regular Solid Army Caps-Olive W35S45D (...    MG         10.99   
4              Washed Canvas Ivy Cap - Black W11S64C    MG         15.99   

  department                               sku  distribution_center_id  
0    

In [13]:
orders=pd.read_csv(os.path.join(klasor_yolu, "orders.csv"))
orders.head()

,order_id,user_id,status,gender,created_at,returned_at,shipped_at,delivered_at,num_of_item
0,9,10,Cancelled,F,2021-12-12 05:47:00 UTC,NaN,NaN,NaN,1
1,37,31,Cancelled,F,2021-01-15 06:22:00 UTC,NaN,NaN,NaN,2
2,44,37,Cancelled,F,2023-12-07 04:25:00 UTC,NaN,NaN,NaN,1
3,56,47,Cancelled,F,2024-04-20 13:21:00 UTC,NaN,NaN,NaN,2
4,65,51,Cancelled,F,2022-08-08 04:57:00 UTC,NaN,NaN,NaN,1


In [15]:
order_items=pd.read_csv(os.path.join(klasor_yolu, "order_items.csv"))
order_items.head(3)

,id,order_id,user_id,product_id,inventory_item_id,status,created_at,shipped_at,delivered_at,returned_at,sale_price
0,89648,61749,49362,14235,241929,Complete,2019-08-04 14:26:00 UTC,2019-08-01 04:33:00 UTC,2019-08-02 09:24:00 UTC,NaN,0.02
1,137052,94346,75385,14159,369603,Cancelled,2024-08-22 16:48:19 UTC,NaN,NaN,NaN,0.49
2,22367,15431,12367,14159,60413,Complete,2020-05-29 12:02:20 UTC,2020-05-29 03:43:00 UTC,2020-05-29 14:09:00 UTC,NaN,0.49


In [16]:
products = pd.read_csv(os.path.join(klasor_yolu, "products_old.csv"))
products.head(3)

,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id
0,13842,2.51875,Accessories,Low Profile Dyed Cotton Twill Cap - Navy W39S55D,MG,6.25,Women,EBD58B8A3F1D72F4206201DA62FB1204,1
1,13928,2.33835,Accessories,Low Profile Dyed Cotton Twill Cap - Putty W39S55D,MG,5.95,Women,2EAC42424D12436BDD6A5B8A88480CC3,1
2,14115,4.87956,Accessories,Enzyme Regular Solid Army Caps-Black W35S45D,MG,10.99,Women,EE364229B2791D1EF9355708EFF0BA34,1


In [22]:
order_items['created_at'] =pd.to_datetime(order_items['created_at'], errors='coerce',utc=True)
order_items['year_month'] =order_items['created_at'].dt.to_period('M').astype(str)
print(order_items.dtypes)

id                                 int64
order_id                           int64
user_id                            int64
product_id                         int64
inventory_item_id                  int64
status                            object
created_at           datetime64[ns, UTC]
shipped_at                        object
delivered_at                      object
returned_at                       object
sale_price                       float64
year_month                        object
dtype: object


In [ ]:
# TABLO 1: AYLIK BÜTÇE VE MALİYET HEDEFLERİ (monthly_budget_targets.csv)

In [29]:
#products tablosundaki 'id' sütununun adını 'product_id' yapıyoruz
products_temiz =products[['id','department', 'category', 'cost']].rename(columns={'id': 'product_id'})
birlesik_tablo =pd.merge(order_items, products_temiz, on='product_id')
print(birlesik_tablo)
birleşik_tablo.info()

            id  order_id  user_id  product_id  inventory_item_id     status  \
0        89648     61749    49362       14235             241929   Complete   
1       137052     94346    75385       14159             369603  Cancelled   
2        22367     15431    12367       14159              60413   Complete   
3        40738     28141    22516       14159             109957   Complete   
4        58354     40241    32110       14159             157594   Complete   
...        ...       ...      ...         ...                ...        ...   
181750  111901     77056    61515       23546             302018    Shipped   
181751  139120     95782    76568       24447             375211    Shipped   
181752  153067    105314    84244       23546             412971    Shipped   
181753  176354    121384    97025       23546             475828    Shipped   
181754  177145    121931    97488       24447             477978    Shipped   

                      created_at               ship

In [31]:
#Her ay ve kategoride toplam ne kadar satış ve maliyet olmuş topluyoruz (Pivot Table gibi)
aylık_özet=birleşik_tablo.groupby(['year_month','department','category'],
                                  as_index=False).agg({'sale_price':'sum', 'cost':'sum'})
aylık_özet.head()

,year_month,department,category,sale_price,cost
0,2019-01,Men,Accessories,22.000000,9.76800
1,2019-01,Men,Active,15.000000,6.16500
2,2019-01,Men,Outerwear & Coats,408.139999,181.08998
3,2019-01,Men,Pants,158.000000,73.66700
4,2019-01,Men,Socks,12.000000,7.24800


In [33]:
#Gerçek satışın %95'i ile %110'u arasında rastgele bir Bütçe Hedefi belirliyoruz
rastgele_oran =np.random.uniform(0.95, 1.10, size=len(aylık_özet))
aylık_özet['budget_revenue_target'] =(aylık_özet['sale_price'] * rastgele_oran).round(2)

In [35]:
#Maliyet limiti ve pazarlama bütçesini de basit birer oranla ekliyoruz
aylık_özet['budget_cost_limit']=(aylık_özet['cost'] * 1.02).round(2)
aylık_özet['marketing_budget_allocated'] =(aylık_özet['budget_revenue_target'] * 0.08).round(2)
budget_df=aylık_özet[['year_month','department', 'category', 'budget_revenue_target', 'budget_cost_limit', 'marketing_budget_allocated']]
budget_df.to_csv(os.path.join(klasor_yolu, "monthly_budget_targets.csv"), index=False)

In [36]:
# 1. Kupon kodlarını ve indirim oranlarını tanımlıyoruz
kupon_kodlari = ['NO_VOUCHER', 'WELCOME10', 'SEASON15', 'VIP25']
olasiliklar   = [0.65, 0.15, 0.12, 0.08] # Toplamı 1.00 (%100) yapıyor
secilen_kuponlar = np.random.choice(kupon_kodlari, size=len(orders), p=olasiliklar)

#Kupon koduna göre indirim oranını eşleştiriyoruz
indirim_sozlugu = {'NO_VOUCHER': 0.0, 'WELCOME10': 0.10, 'SEASON15': 0.15, 'VIP25': 0.25}
indirim_oranlari = pd.Series(secilen_kuponlar).map(indirim_sozlugu)

# Kargo Maliyeti = Sabit 5$ + (Ürün Adedi x 1.25$)
kargo_maliyeti = (5.0 + (orders['num_of_item'] * 1.25)).round(2)
iade_maliyeti = np.where(orders['status'] == 'Returned', kargo_maliyeti, 0.0)
vouchers_shipping_df = pd.DataFrame({
    'order_id': orders['order_id'],
    'voucher_code': secilen_kuponlar,
    'voucher_discount_rate': indirim_oranlari,
    'shipping_cost': kargo_maliyeti,
    'return_logistics_cost': iade_maliyeti
})

vouchers_shipping_df.to_csv(os.path.join(klasor_yolu, "order_vouchers_shipping.csv"), index=False)